# M05 — Generator pipeline proof (Qwen3.5-0.8B): zero-shot check + SFT

**This run is the 0.8B PIPELINE PROOF, not a method comparison.** Per D05, the M06
ordinary-generation baseline and the RL arm must use the SAME model (Qwen3.5-2B);
numbers from this 0.8B run feed no cross-model or cross-method claims.

Run top-to-bottom on **Google Colab** (T4 GPU, `Runtime -> Run all`, ~60 min; models
and dataset come from the Drive cache on re-runs).

Stages:

1. **Bank lock** — rebuild the frozen Option B request bank (200 requests, seed
   20260920) from the pinned pools; assert its sha256 and the final-eval lock.
2. **Zero-shot instruction-following** — all 200 bank requests through
   Qwen3.5-0.8B (text-only, non-thinking, greedy). Contract = JSON object with a
   `text` field, target name exactly once (word-boundary matching).
3. **SFT** — LoRA on dataset-derived single-name pairs (train-side rows only,
   final-eval excluded). Examples are tokenized from the FULL user+assistant
   conversation via `apply_chat_template`; the label is verified to end with the
   real end-of-turn token. Pairs capped at 2000, sequences at 512 tokens.
   Assistant-token loss only.
4. **Post-SFT re-check** — same 200 requests; zero-shot vs SFT contract rates.
5. **Diversity check** — sampled generation (temperature 0.8, n=4 per request) on
   50 bank requests: duplicate rate, distinct-1/2, output length.
6. **Frozen-D0 flagging** — the frozen detector D0 (verified by sha256) flags
   PERSON spans beyond the requested name in generated texts. FLAG ONLY: this is
   a quality signal for review, never a reward component (M07 wires rewards
   separately, and flag-only here keeps the seam honest).

Everything (adapter, report, failures) is saved to Drive under
`MyDrive/pii-redteam-rl/m05-<timestamp>/`.

In [ ]:
import datetime
import json
import os
import re
import sys

REPO_URL = "https://github.com/GauthierRoy/pii-redteam-rl.git"
REPO_DIR = "/content/pii-redteam-rl"
DRIVE_ROOT = "/content/drive/MyDrive/pii-redteam-rl"
RUN_ID = datetime.datetime.now(datetime.UTC).strftime("m05-%Y%m%d-%H%M%S")
WORK = os.path.join(DRIVE_ROOT, RUN_ID)
CACHE = os.path.join(DRIVE_ROOT, "cache")
EXPECTED_FINAL_EVAL_SHA = "513335badf5752a340698d3c7efa7029a8780c51fde2675c80ef4208ce27d39c"
EXPECTED_BANK_SHA = "332a9eb0b682bc7213aa556290d1eeb9f616be81b3146c382bcf9c37300dc58c"
D0_PATH = os.path.join(DRIVE_ROOT, "d0-20261004-142714", "d0-best")
EXPECTED_D0_SHA = "0314ebe36d7f4dc6d784f0e139ff5b9d32633aef1a6f88e764a610a8d8ec43cb"

SEED = 20260920
BANK_N = 200
GEN_MAX_NEW_TOKENS = 150
SFT_MAX_PAIRS = 2000
SFT_MAX_TOKENS = 512
SFT_BATCH = 8
EPOCHS = 3
LR = 1e-4
LORA_R = 16
SAMPLE_REQUESTS = 50
SAMPLE_N = 4
SAMPLE_TEMP = 0.8

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
os.makedirs(CACHE, exist_ok=True)

REPORT = {
    "run_id": RUN_ID,
    "role": (
        "0.8B pipeline proof; M06 baseline must use the same model as the RL arm "
        "(Qwen3.5-2B per D05); results not comparable across model sizes"
    ),
    "checks": {},
    "config": {
        "seed": SEED, "bank_n": BANK_N, "max_new_tokens": GEN_MAX_NEW_TOKENS,
        "sft_max_pairs": SFT_MAX_PAIRS, "sft_max_tokens": SFT_MAX_TOKENS,
        "batch": SFT_BATCH, "epochs": EPOCHS, "lr": LR, "lora_r": LORA_R,
        "sample_requests": SAMPLE_REQUESTS, "sample_n": SAMPLE_N, "sample_temp": SAMPLE_TEMP,
    },
}


def check(name, ok, detail=""):
    REPORT["checks"][name] = {"ok": bool(ok), "detail": str(detail)}
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f" -- {detail}" if detail else ""))


print(f"run_id: {RUN_ID}  ->  {WORK}")

In [ ]:
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
REPO_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "peft", "torchao"],
    check=True,
)
import torch  # noqa: E402
import transformers  # noqa: E402

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from pii_redteam.data import QuarantinedRecord, build_sft_pair, is_single_person_once, to_canonical  # noqa: E402
from pii_redteam.detector import spans_from_bio_labels, trim_span_whitespace  # noqa: E402
from pii_redteam.requests import render_prompt  # noqa: E402

torch.manual_seed(SEED)
REPORT["environment"] = {
    "repo_commit": REPO_COMMIT, "python": sys.version.split()[0],
    "torch": torch.__version__, "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
}
print(json.dumps(REPORT["environment"], indent=2))
assert torch.cuda.is_available(), "GPU runtime required."

In [ ]:
import hashlib

# Rebuild the locked splits, then the frozen request bank; assert both locks.
splits_dir = os.path.join(REPO_DIR, "artifacts", "splits")
subprocess.run(
    [sys.executable, "scripts/carve_splits.py", "--cache-dir", CACHE, "--out", splits_dir],
    cwd=REPO_DIR, env=dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src")), check=True,
)
manifest = json.load(open(os.path.join(splits_dir, "final_eval_manifest.json")))
check("final-eval lock reproduces", manifest["final_eval_sha256"] == EXPECTED_FINAL_EVAL_SHA,
      manifest["final_eval_sha256"])
if manifest["final_eval_sha256"] != EXPECTED_FINAL_EVAL_SHA:
    raise SystemExit("Lock mismatch.")

subprocess.run(
    [sys.executable, "scripts/build_request_bank.py", "--n", str(BANK_N), "--seed", str(SEED)],
    cwd=REPO_DIR, env=dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src")), check=True,
)
bank_path = os.path.join(REPO_DIR, "artifacts", "request_bank", f"sampler_seed{SEED}_n{BANK_N}.jsonl")
bank_sha = hashlib.sha256(open(bank_path, "rb").read()).hexdigest()
check("request bank reproduces (frozen lock)", bank_sha == EXPECTED_BANK_SHA, bank_sha)
REQUESTS = [json.loads(line) for line in open(bank_path, encoding="utf-8")]
print(f"bank: {len(REQUESTS)} requests")

# Dataset-derived SFT pairs (capped): single-person-once train-side rows only.
FINAL_EVAL_IDS = {r["example_id"] for r in manifest["rows"]}
pairs = []
with open(os.path.join(CACHE, "1english_openpii_30k.jsonl"), encoding="utf-8") as f:
    for line in f:
        if len(pairs) >= SFT_MAX_PAIRS:
            break
        try:
            canonical = to_canonical(json.loads(line))
        except QuarantinedRecord:
            continue
        if canonical["example_id"] in FINAL_EVAL_IDS or not is_single_person_once(canonical):
            continue
        request, target = build_sft_pair(canonical, request_id=f"sft-{len(pairs):04d}")
        pairs.append({"request": request, "target": target})
print(f"SFT pairs (cap {SFT_MAX_PAIRS}): {len(pairs)}")
REPORT["sft_pairs_built"] = len(pairs)

In [ ]:
import time

from transformers import AutoModelForCausalLM, AutoModelForTokenClassification, AutoTokenizer

QWEN_ID = "Qwen/Qwen3.5-0.8B"
tok = AutoTokenizer.from_pretrained(QWEN_ID)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(QWEN_ID, dtype=dtype, device_map="cuda")
check("Qwen3.5-0.8B text-only load", True, f"AutoModelForCausalLM dtype={dtype}")

# Frozen D0 (sha-verified) for extra-PERSON FLAGGING ONLY - never a reward component.
D0 = None
d0_tok = None
d0_weights = os.path.join(D0_PATH, "model.safetensors")
if os.path.exists(d0_weights):
    d0_sha = hashlib.sha256(open(d0_weights, "rb").read()).hexdigest()
    check("frozen D0 checkpoint verified", d0_sha == EXPECTED_D0_SHA, d0_sha[:16])
    d0_tok = AutoTokenizer.from_pretrained(D0_PATH)
    D0 = AutoModelForTokenClassification.from_pretrained(D0_PATH).cuda()
    D0.eval()
else:
    check("frozen D0 checkpoint verified", False,
          f"{D0_PATH} not found on Drive - extra-PERSON flagging DISABLED")


def chat_text_for(request):
    try:
        return tok.apply_chat_template(
            [{"role": "user", "content": render_prompt(request)}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
    except TypeError:
        return tok.apply_chat_template(
            [{"role": "user", "content": render_prompt(request)}],
            tokenize=False, add_generation_prompt=True,
        )


@torch.no_grad()
def generate_batch(requests, batch_size=16, **gen_kwargs):
    """Batched generation; returns list of decoded continuation strings."""
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(requests), batch_size):
        chunk = requests[i : i + batch_size]
        texts = [chat_text_for(r) for r in chunk]
        enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=GEN_MAX_NEW_TOKENS, **gen_kwargs)
        for j in range(len(chunk)):
            outs.append(tok.decode(gen[j][enc["input_ids"].shape[1]:], skip_special_tokens=True))
        if (i // batch_size) % 4 == 0:
            print(f"  generated {i + len(chunk)}/{len(requests)}")
    return outs


def extract_text(output):
    """Parse the JSON output contract; return (text_or_None, json_ok)."""
    try:
        parsed = json.loads(output)
        text = parsed.get("text") if isinstance(parsed, dict) else None
        return (text, isinstance(text, str))
    except json.JSONDecodeError:
        return (None, False)


def count_name(text, name):
    """Word-boundary occurrence count of the exact name in the text."""
    return len(re.findall(rf"(?<!\w){re.escape(name)}(?!\w)", text))


@torch.no_grad()
def d0_person_spans(texts):
    """Frozen-D0 PERSON spans per text; [] when D0 is unavailable."""
    if D0 is None:
        return [[] for _ in texts]
    out = []
    for i in range(0, len(texts), 32):
        chunk = texts[i : i + 32]
        enc = d0_tok(chunk, return_offsets_mapping=True, truncation=True,
                     max_length=512, padding=True)
        input_ids = torch.tensor(enc["input_ids"]).cuda()
        attn = torch.tensor(enc["attention_mask"]).cuda()
        logits = D0(input_ids=input_ids, attention_mask=attn).logits
        for j, text in enumerate(chunk):
            offsets = enc["offset_mapping"][j]
            labels = [D0.config.id2label[p] for p in logits[j].argmax(-1).tolist()]
            out.append(trim_span_whitespace(text, spans_from_bio_labels(offsets, labels)))
    return out


def flag_extra_person(requests, outputs):
    """Frozen-D0 extra-PERSON flags: predicted spans not overlapping the requested
    name occurrence. FLAG ONLY - informational, never a reward."""
    if D0 is None:
        return None
    span_lists = d0_person_spans(outputs)
    flagged = []
    for request, text, spans in zip(requests, outputs, span_lists, strict=True):
        if text is None:
            continue
        name = request["task"]["entities"][0]["value"]
        matches = list(re.finditer(rf"(?<!\w){re.escape(name)}(?!\w)", text))
        extra = [
            {"value": text[sp["start"]:sp["end"]], "start": sp["start"], "end": sp["end"]}
            for sp in spans
            if not any(m.start() < sp["end"] and m.end() > sp["start"] for m in matches)
        ]
        flagged.extend({"request_id": request["request_id"], "extra": e} for e in extra)
    return {"texts": len(outputs), "extra_person_flags": len(flagged), "detail": flagged[:20]}


def contract_results(requests, outputs):
    ok_json = ok_once = 0
    failures = []
    for request, out in zip(requests, outputs, strict=True):
        name = request["task"]["entities"][0]["value"]
        text, json_ok = extract_text(out)
        ok_json += json_ok
        once = bool(text) and count_name(text, name) == 1
        ok_once += once
        if not (json_ok and once) and len(failures) < 20:
            failures.append({
                "request_id": request["request_id"], "name": name,
                "output": out[:400], "json_ok": json_ok,
                "name_occurrences": count_name(text, name) if text else -1,
            })
    return {"pass_json": ok_json, "pass_once": ok_once, "total": len(requests),
            "rate_json": round(ok_json / len(requests), 4),
            "rate_once": round(ok_once / len(requests), 4), "failures": failures}


t0 = time.time()
zeroshot_outputs = generate_batch(REQUESTS, do_sample=False)
zeroshot_s = round(time.time() - t0, 1)
zero = contract_results(REQUESTS, zeroshot_outputs)
zero_flags = flag_extra_person(REQUESTS, zeroshot_outputs)
REPORT["zeroshot"] = {k: zero[k] for k in ("pass_json", "pass_once", "total", "rate_json", "rate_once")}
REPORT["zeroshot_d0_flags"] = None if zero_flags is None else {
    "texts": zero_flags["texts"], "extra_person_flags": zero_flags["extra_person_flags"]}
print(f"zero-shot: JSON {zero['rate_json']:.0%}, name-exactly-once(word-boundary) "
      f"{zero['rate_once']:.0%} ({zeroshot_s}s)")
check("zero-shot instruction-following measured", True,
      f"json_ok={zero['rate_json']:.2f} name_once={zero['rate_once']:.2f} (informational, no floor)")
if zero_flags:
    print(f"frozen-D0 extra-PERSON flags on zero-shot outputs: {zero_flags['extra_person_flags']}")
print(json.dumps(zero["failures"][:3], indent=2, ensure_ascii=False))

In [ ]:
import math

import time

from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader, Dataset

# SFT examples: apply_chat_template over the FULL user+assistant conversation;
# prompt tokens are masked via the common prefix, assistant tokens carry the loss.
# NOTE (transformers 5.x): apply_chat_template(tokenize=True) returns a BatchEncoding,
# not a list of ids - normalize before any comparison (verified locally: prefix match
# is 100% and the label tail ends with the real end-of-turn token <|im_end|>).
def template_ids(messages, **kwargs):
    try:
        out = tok.apply_chat_template(messages, tokenize=True, enable_thinking=False, **kwargs)
    except TypeError:
        out = tok.apply_chat_template(messages, tokenize=True, **kwargs)
    ids = out["input_ids"] if hasattr(out, "keys") else out
    if ids and isinstance(ids[0], list):
        ids = ids[0]
    return list(ids)


def build_sft_example(request, target):
    user = [{"role": "user", "content": render_prompt(request)}]
    full = user + [{"role": "assistant", "content": json.dumps({"text": target}, ensure_ascii=False)}]
    prefix = template_ids(user, add_generation_prompt=True)
    ids = template_ids(full, add_generation_prompt=False)
    k = 0
    while k < min(len(prefix), len(ids)) and prefix[k] == ids[k]:
        k += 1
    if k < int(len(prefix) * 0.9):
        raise ValueError("chat-template prompt prefix mismatch; refusing to train blind")
    return {"input_ids": ids, "labels": [-100] * k + ids[k:]}


# HARD GATE: the label must end with the real end-of-turn token, not a pad and not
# mid-content. Verified on a sample before any training.
probe = build_sft_example(pairs[0]["request"], pairs[0]["target"])
tail = [tok.decode([i]) for i in probe["input_ids"][-3:]]
end_ok = probe["labels"][-1] == probe["input_ids"][-1] and (
    any("im_end" in t or t.strip() == tok.eos_token.strip() for t in tail)
)
check("SFT labels end with the real end-of-turn token", end_ok, f"tail tokens: {tail}")
if not end_ok:
    raise SystemExit("End-of-turn verification failed - refusing to train with bad labels.")

# Length cap: drop pairs whose full conversation exceeds SFT_MAX_TOKENS.
sft_items, skipped_long = [], 0
for item in pairs:
    ex = build_sft_example(item["request"], item["target"])
    if len(ex["input_ids"]) > SFT_MAX_TOKENS:
        skipped_long += 1
        continue
    sft_items.append(ex)
print(f"SFT items: {len(sft_items)} (dropped {skipped_long} over {SFT_MAX_TOKENS} tokens)")
REPORT["sft_items"] = len(sft_items)
REPORT["sft_skipped_long"] = skipped_long


class SftDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def sft_collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    input_ids, labels, attn = [], [], []
    for b in batch:
        n = maxlen - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [tok.pad_token_id] * n)
        labels.append(b["labels"] + [-100] * n)
        attn.append([1] * len(b["input_ids"]) + [0] * n)
    return {"input_ids": torch.tensor(input_ids), "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(attn)}


linear_names = [n.split(".")[-1] for n, m in model.named_modules() if isinstance(m, torch.nn.Linear)]
targets = sorted({n for n in linear_names if n in {"q_proj", "k_proj", "v_proj", "o_proj"}}) or ["q_proj"]
model = get_peft_model(
    model,
    LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, target_modules=targets,
               lora_dropout=0.0, task_type="CAUSAL_LM"),
)
model.print_trainable_parameters()

loader = DataLoader(SftDataset(sft_items), batch_size=SFT_BATCH, shuffle=True,
                    collate_fn=sft_collate)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
t0 = time.time()
losses = []
model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss, n = 0.0, 0
    for batch in loader:
        with torch.autocast("cuda", dtype=dtype):
            loss = model(
                input_ids=batch["input_ids"].cuda(),
                attention_mask=batch["attention_mask"].cuda(),
                labels=batch["labels"].cuda(),
            ).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n += 1
    losses.append(round(epoch_loss / n, 4))
    print(f"epoch {epoch}: mean loss {losses[-1]}")
sft_s = round(time.time() - t0, 1)
check("SFT ran", all(math.isfinite(l) for l in losses) and len(losses) == EPOCHS,
      f"epoch losses={losses} ({sft_s}s)")
model.save_pretrained(os.path.join(WORK, "qwen-sft-lora"))
tok.save_pretrained(os.path.join(WORK, "qwen-sft-lora"))
trainable, total_params = model.get_nb_trainable_parameters()
REPORT["sft"] = {"targets": targets, "epoch_losses": losses, "seconds": sft_s,
                 "trainable_params": trainable, "total_params": total_params}
print(f"adapter saved to {WORK}/qwen-sft-lora")

# Post-SFT re-check on the same frozen bank.
model.eval()
t0 = time.time()
sft_outputs = generate_batch(REQUESTS, do_sample=False)
sft_gen_s = round(time.time() - t0, 1)
post = contract_results(REQUESTS, sft_outputs)
post_flags = flag_extra_person(REQUESTS, sft_outputs)
REPORT["post_sft"] = {k: post[k] for k in ("pass_json", "pass_once", "total", "rate_json", "rate_once")}
REPORT["post_sft_d0_flags"] = None if post_flags is None else {
    "texts": post_flags["texts"], "extra_person_flags": post_flags["extra_person_flags"]}
print(f"post-SFT: JSON {post['rate_json']:.0%}, name-exactly-once {post['rate_once']:.0%} ({sft_gen_s}s)")
if post_flags:
    print(f"frozen-D0 extra-PERSON flags on post-SFT outputs: {post_flags['extra_person_flags']}")
check("SFT improved or held the contract rate",
      post["rate_once"] >= zero["rate_once"],
      f"zero-shot once-rate={zero['rate_once']} post-SFT once-rate={post['rate_once']}")

In [ ]:
# Diversity check: sampled generation (temp 0.8, n=4) on the first SAMPLE_REQUESTS
# bank requests. Reports duplicate rate, word-level distinct-1/2, output length.
@torch.no_grad()
def sampled_diversity(requests):
    tok.padding_side = "left"
    reqs = requests[:SAMPLE_REQUESTS]
    dup_rates, d1s, d2s, lens = [], [], [], []
    model.eval()
    for i in range(0, len(reqs), 8):
        chunk = reqs[i : i + 8]
        texts = [chat_text_for(r) for r in chunk]
        enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(
            **enc, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=True,
            temperature=SAMPLE_TEMP, top_p=0.95, num_return_sequences=SAMPLE_N,
        )
        for j in range(len(chunk)):
            outs = [
                tok.decode(gen[j * SAMPLE_N + k][enc["input_ids"].shape[1]:],
                           skip_special_tokens=True)
                for k in range(SAMPLE_N)
            ]
            dup_rates.append(1 - len(set(outs)) / SAMPLE_N)
            word_sets = [o.split() for o in outs]
            words = [w for ws in word_sets for w in ws]
            d1s.append(len(set(words)) / max(len(words), 1))
            bigrams = [" ".join(ws[b : b + 2]) for ws in word_sets for b in range(len(ws) - 1)]
            d2s.append(len(set(bigrams)) / max(len(bigrams), 1))
            lens.extend(len(o) for o in outs)
    return {
        "requests": len(reqs), "n": SAMPLE_N, "temp": SAMPLE_TEMP,
        "duplicate_rate": round(sum(dup_rates) / len(dup_rates), 4),
        "distinct1": round(sum(d1s) / len(d1s), 4),
        "distinct2": round(sum(d2s) / len(d2s), 4),
        "mean_output_chars": round(sum(lens) / len(lens), 1),
    }


t0 = time.time()
diversity = sampled_diversity(REQUESTS)
diversity["seconds"] = round(time.time() - t0, 1)
REPORT["sampled_diversity"] = diversity
print(json.dumps(diversity, indent=2))
check("sampled diversity measured (informational)", True,
      f"duplicate_rate={diversity['duplicate_rate']} distinct1={diversity['distinct1']} "
      f"distinct2={diversity['distinct2']} mean_chars={diversity['mean_output_chars']}")

In [ ]:
REPORT["finished_utc"] = datetime.datetime.now(datetime.UTC).isoformat()
REPORT["zeroshot_failures"] = zero["failures"]
REPORT["post_sft_failures"] = post["failures"]
if zero_flags:
    REPORT["zeroshot_d0_flag_detail"] = zero_flags["detail"]
if post_flags:
    REPORT["post_sft_d0_flag_detail"] = post_flags["detail"]

report_path = os.path.join(WORK, "m05_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(REPORT, f, indent=2, ensure_ascii=False)

print("=" * 70)
print("M05 GENERATOR SUMMARY - 0.8B PIPELINE PROOF (not a method comparison)")
print("=" * 70)
for name, res in REPORT["checks"].items():
    print(f"[{'PASS' if res['ok'] else 'FAIL'}] {name}")
    if res["detail"]:
        print(f"       {res['detail']}")
print("=" * 70)
print(f"adapter: {WORK}/qwen-sft-lora")
print(f"report:  {report_path}")
print("Next: bring back m05_report.json. M06 baseline runs this SAME pipeline on")
print("Qwen3.5-2B (the RL arm's model, per D05) - do not compare 0.8B vs 2B numbers.")
if not all(r["ok"] for r in REPORT["checks"].values()):
    print("NOTE: some checks FAILED - record them honestly; do not retune silently.")